# Table RAG benchmark: full run on a GPU

Runs every ingestion method (fixed-size chunks, per-page markdown, Camelot, Docling, ours) on both benchmarks
(SpecQA and T²-RAGBench), with hybrid search (BM25 + `bge-small`) and `Qwen2.5-3B-Instruct` as the reader.

**Before you start:** *Runtime → Change runtime type → T4 GPU → Save*, then *Runtime → Run all*.
About 45–60 minutes. The last cell downloads `table_rag_results.zip`.

## 1. Check the GPU

In [ ]:
import torch
print("PyTorch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU, then run this cell again."

## 2. Get the code and install (≈ 3 min)

In [ ]:
BRANCH = "main"   # or the branch you want to test
%cd /content
!rm -rf RAG
!git clone -q -b {BRANCH} https://github.com/namita-tj/RAG.git
%cd /content/RAG
!git log --oneline -1
!apt-get -qq install -y fonts-dejavu-core > /dev/null
!pip install -q -e ".[gpu,baselines]"
import os
assert os.path.exists("src/tablerag/bench/run.py"), "tablerag is missing: check BRANCH."

## 3. Run the benchmark (≈ 45–60 min)

The T²-RAGBench questions are downloaded from Hugging Face (`G4KMU/t2-ragbench`, CC-BY-4.0); 100 contexts per subset
are rendered into PDFs. Progress is printed per method.

In [ ]:
!python -m tablerag.bench.run --suite specqa t2 --retriever hybrid --reader Qwen/Qwen2.5-3B-Instruct \
    --methods naive markdown camelot docling ours ablate_no_context ablate_no_stitch ablate_rows_only \
    --t2-contexts 100 --data benchmarks/data --out results/gpu

## 4. Results

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("results/gpu/summary.md").read()))

## 5. Download the results (`table_rag_results.zip`)

In [ ]:
!rm -f table_rag_results.zip && zip -q -r table_rag_results.zip results/gpu && ls -la table_rag_results.zip
from google.colab import files
files.download("table_rag_results.zip")